In [6]:
%pip install jinja2
import os
import glob
import re
import pandas as pd
from datetime import datetime

# 🟢 1. 強制解除 Pandas 欄位寬度與顯示限制（確保長門牌、長訊息不被切字）
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

def view_live_logs(log_type="all", limit=50):
    
    LOG_DIR = "../logs"
    if not os.path.exists(LOG_DIR):
        print(f"❌ 找不到日誌目錄：{os.path.abspath(LOG_DIR)}")
        return
    
    # 精準定位今天的日期後綴 (YYYYMMDD)
    today_str = datetime.now().strftime("%Y%m%d")
    
    # 優先抓取今日特定日誌檔案，維持收集器的高效能
    log_files = []
    if log_type in ["crawler", "all"]:
        crawler_log = os.path.join(LOG_DIR, f"crawler_{today_str}.log")
        if os.path.exists(crawler_log): log_files.append(crawler_log)
        
    if log_type in ["api", "all"]:
        api_log = os.path.join(LOG_DIR, f"api_{today_str}.log")
        if os.path.exists(api_log): log_files.append(api_log)
    
    # 🔄 保底機制：如果今天還沒產生 Log，就退回使用 glob 抓取所有歷史 Log 檔案
    if not log_files:
        log_files = glob.glob(os.path.join(LOG_DIR, "*.log"))
        
    if not log_files:
        print("📭 目前還沒有任何 Log 日誌檔案產生。")
        return
    
    all_logs_parsed = []
    
    # 💡 定義標準 Log 刀模：抓取開頭的時間、中間中括號裡的等級、後面的所有訊息
    # 格式舉例: 2026-05-24 20:09:48,536 [INFO] 💾 [CSV 匯出成功]
    log_pattern = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3})\s*\[(.*?)\]\s*(.*)$")
  
    for file_path in log_files:
        file_name = os.path.basename(file_path)
        if log_type == "crawler" and "crawler" not in file_name: continue
        if log_type == "api" and "api" not in file_name: continue
            
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                clean_line = line.strip()
                if not clean_line: continue
                
                # 用 Regex 進行精準匹配
                match = log_pattern.match(clean_line)
                
                if match:
                    # 🎯 匹配成功
                    timestamp = match.group(1)
                    level = match.group(2).strip()
                    message = match.group(3)
                    
                    all_logs_parsed.append({
                        "時間": timestamp,
                        "來源檔案": file_name,
                        "日誌等級": level,
                        "詳細訊息內容": message
                    })
                else:
                    # 💥 匹配失敗
                    if all_logs_parsed:
                        all_logs_parsed[-1]["詳細訊息內容"] += f" {clean_line}"

    if not all_logs_parsed:
        print(f"📭 沒有符合條件 ({log_type}) 的日誌紀錄。")
        return

    # 轉換成 DataFrame
    log_df = pd.DataFrame(all_logs_parsed)
    log_df = log_df.sort_values(by="時間", ascending=False).reset_index(drop=True).head(limit)
    
    print(f"📊 【Log 收集監控面板】當前即時在線日誌總量: {len(log_df)} 筆 (展示最新 {limit} 筆)")
    print("-" * 80)
    
    # ─── 網格網頁級排版 ───
    styled_panel = log_df.style.set_properties(**{
        'background-color': '#fafafa',
        'border-color': '#e0e0e0',
        'border-style': 'solid',
        'border-width': '1px',
        'padding': '6px 12px'
    })
    
    
    # 隱藏檔名欄位，釋放極致寬度
    if '來源檔案' in log_df.columns:
        styled_panel = styled_panel.hide(['來源檔案'], axis='columns')
        
    # 動態色彩
    def apply_log_color(val):
        if val in ['ERROR', '[ALERT]', 'ERROR ']:
            return 'color: #d32f2f; font-weight: bold; background-color: #ffebee;'
        elif val in ['WARNING', 'WARNING ']:
            return 'color: #f57c00; font-weight: bold; background-color: #fff3e0;'
        elif val in ['INFO', 'INFO ']:
            return 'color: #388e3c;'
        return ''

    # 歷史版本相容性防禦
    if hasattr(styled_panel, 'map'):
        return styled_panel.map(apply_log_color, subset=['日誌等級'])
    else:
        return styled_panel.applymap(apply_log_color, subset=['日誌等級'])

# ── 💡 重新測試 ──
view_live_logs(log_type="all")


📊 【Log 收集監控面板】當前即時在線日誌總量: 50 筆 (展示最新 50 筆)
--------------------------------------------------------------------------------


You should consider upgrading via the 'C:\Users\USER\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


,時間,日誌等級,詳細訊息內容
0,"2026-05-25 03:29:00,764",WARNING,[ALERT] 查詢結果為空 city=台北縣 township=大安區 —— 觸發異常通知
1,"2026-05-25 03:29:00,760",INFO,收到查詢請求 city=台北縣 township=大安區
2,"2026-05-25 03:27:57,907",WARNING,[ALERT] 查詢結果為空 city=台北市 township=天母區 —— 觸發異常通知
3,"2026-05-25 03:27:57,901",INFO,收到查詢請求 city=台北市 township=天母區
4,"2026-05-25 03:27:40,693",INFO,查詢成功，回傳 130 筆資料
5,"2026-05-25 03:27:40,669",INFO,收到查詢請求 city=台北市 township=大安區
6,"2026-05-25 03:23:37,410",INFO,💾 [Pipeline 結束] 成功落檔 CSV 檔案並完整寫入 SQLite 資料庫！
7,"2026-05-25 03:23:37,409",INFO,💾 [Database 寫入成功] 已成功將資料追加至 C:\Users\USER\Desktop\Prepare\2026\Untitled Folder\cathay-ins Project\taiwan_address.db 檔案中！
8,"2026-05-25 03:23:37,375",INFO,💾 [Database] 正在將清洗後的 DataFrame 追加寫入資料庫...
9,"2026-05-25 03:23:37,374",INFO,💾 [CSV 匯出成功] 實體檔案已寫入至本地磁碟：臺北市門牌大數據總表_已清洗.csv
